# 17장 · 임베딩으로 설명서 문단 찾기

“앞쪽 빨간불이 안 보이게 할 수 있나요?”라는 질문에는 ‘표시등’이란 단어가 없습니다. 문서 안에서 같은 글자를 찾는 방법은 이 질문을 놓칠 수 있습니다. 표현이 다른 질문을 같은 설명 쪽으로 연결하는 검색기를 만들고, 가까이 나왔지만 틀린 기종인 후보까지 읽습니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 글을 숫자 목록으로 바꾸기

임베딩 모델은 글 한 덩어리를 숫자들의 목록, 즉 벡터로 바꿉니다. 모델이 같은 방식으로 만든 벡터끼리 거리를 비교해 비슷한 용법의 문단을 찾을 수 있습니다. 어느 좌표 하나가 ‘빨간불 점수’라는 뜻은 없습니다. 문단과 질문 모두 같은 임베딩 모델을 써야 비교의 기준이 같습니다.

In [2]:
from rag_lab import MANUALS, read_pages, EMBEDDING_MODEL
from langchain_openai import OpenAIEmbeddings

pages = [page for file in MANUALS for page in read_pages(file)]
embed = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector = embed.embed_query("녹음할 때 앞쪽 빨간불을 숨기고 싶어요.")
print("문서 쪽", len(pages), "질문 벡터의 숫자 개수", len(vector))

문서 쪽 6 질문 벡터의 숫자 개수 1536


## 벡터를 문장과 함께 보관하기

Chroma의 컬렉션은 같은 기준으로 검색할 레코드 묶음입니다. 레코드에는 문단의 벡터, 원문, 출처를 함께 넣습니다. 벡터만 받으면 가까운 숫자는 찾을 수 있어도 학생에게 근거 쪽을 보여 줄 수 없습니다. 우선 서로 비슷한 A·B형 표시등 두 쪽과 B형 파일 저장 한 쪽만 넣어 손으로 확인합니다.

In [3]:
from rag_lab import open_index, retrieve
tiny = [p for p in pages if (p["model"], p["edition"], p["page"]) in
        [("A","1.1",1), ("B","1.0",1), ("B","1.0",2)]]
index = open_index(tiny)
query = "녹음 도중 빨간 표시등을 끌 수 있나요?"
for row in retrieve(index, query, k=3):
    print(row["id"], row["text"][:30])

recorder-a-1.1.pdf#p1 들소리 A형   /   휴대용 녹음기
녹음 표시등
1.
recorder-b-1.0.pdf#p1 들소리 B형   /   휴대용 녹음기
녹음 표시등
1.
recorder-b-1.0.pdf#p2 들소리 B형   /   휴대용 녹음기
파일 저장과 복사


목록 맨 위는 모델이 가까이 본 쪽입니다. A형의 “끄는 설정은 없습니다”와 B형의 “숨김으로 바꾸세요”는 반대 행동을 말하면서도 모두 표시등을 설명합니다. **가까운 문단이 맞는 답이라는 보장은 없습니다.** 먼저 질문에 기종이 있는지, 후보에 실제 기종이 적혀 있는지 보세요.

In [4]:
another = "B형으로 소리가 나는 동안 앞쪽 불빛을 안 보이게 하고 싶어요."
for row in retrieve(index, another, k=2):
    print(row["source"], row["page"], row["text"][60:175])

recorder-b-1.0.pdf 1 등 설정
녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸세요. 녹음을
시작해도 앞면에 빨간불이 들어오지 않습니다.
화면과 표시등
이미 녹음 중이면 표시등 설정을 바꿀 수 없습니다. 녹음을 끝낸
뒤 설정에서 ‘
recorder-a-1.1.pdf 1 등 설정
녹음 중에는 앞면의 빨간 표시등이 켜집니다. 녹음하는 동안
표시등을 끄는 설정은 없습니다.
화면과 표시등
화면 밝기를 낮춰도 표시등은 켜져 있습니다. 조용한 방에서
화면을 어둡게 하고 싶을 때는 설


## Chroma에 세 레코드 직접 넣기

`open_index`는 수업 자료를 계속 보관할 때 씁니다. 한 번은 보조 함수 없이 문서 ID·본문·벡터가 어디에 들어가는지 확인하세요. 아래 임시 컬렉션은 디스크에 저장하지 않습니다. 같은 셀을 두 번 실행해도 같은 세 ID를 갱신합니다. 질문의 벡터 한 개를 넣어 첫 질문의 id, 본문, 거리를 함께 받아 봅니다.

In [5]:
import chromadb
demo = chromadb.EphemeralClient().get_or_create_collection("recorder-three-pages")
texts = [r["text"] for r in tiny]
demo.upsert(ids=[r["id"] for r in tiny], documents=texts,
            embeddings=embed.embed_documents(texts))
hits = demo.query(query_embeddings=[embed.embed_query(another)], n_results=3,
                  include=["documents", "distances"])
for key, distance, text in zip(hits["ids"][0], hits["distances"][0], hits["documents"][0]):
    print(key, round(distance, 3), text[:25])

recorder-b-1.0.pdf#p1 1.245 들소리 B형   /   휴대용 녹음기
녹음 표
recorder-a-1.1.pdf#p1 1.331 들소리 A형   /   휴대용 녹음기
녹음 표
recorder-b-1.0.pdf#p2 1.345 들소리 B형   /   휴대용 녹음기
파일 저


바깥 목록의 [0]은 첫 번째 질문의 결과입니다. 그 안의 첫 값이 1위입니다. 여기서 거리가 작을수록 벡터가 가깝지만 ‘정답일 확률 90%’로 해석하지 않습니다. B형 질문의 상위 결과에서 A형 자료가 나왔다면 거리와 근거 문장을 각자 살펴보세요. 사용자가 필요로 하는 조건을 만족하는지 읽는 단계가 남아 있습니다.

용어를 같은 것으로 처리하는 것과 뜻이 비슷한 글을 검색하는 것은 다릅니다. 기종이 A형과 B형으로 한 글자만 달라도 동작은 반대일 수 있습니다. 누구에게나 같은 후보 수를 쓸지 결정하려면 실제 문의의 제품명·시점·행동을 기준으로 후보를 읽어야 합니다. [Chroma 컬렉션과 검색 참고](https://docs.trychroma.com/docs/querying-collections/query-and-get)

## 실습 · 표현을 바꾸어도 정답 쪽이 들어오는가

기준 색인을 세 PDF 전체로 엽니다. 2027년 5월 A형의 ‘표시등’ 질문과 직접 쓴 ‘빨간불’ 질문을 검색해 정답 페이지의 순위를 적습니다. 질문에서 기종을 지운 뒤 또 검색하세요. 순위가 같아도 답을 확정할 수 있는 정보가 그대로인지 설명하세요.

In [6]:
# open_index()로 전체 자료를 열고 같은 k로 검색하세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>full_index = open_index()
for q in [&quot;2027년 5월 A형 녹음 표시등을 끌 수 있나요?&quot;,
          &quot;2027년 5월 A형 녹음 중 앞의 빨간불을 감출 수 있나요?&quot;,
          &quot;녹음 중에 앞쪽 불을 감출 수 있나요?&quot;]:
    print(q, [p[&quot;id&quot;] for p in retrieve(full_index, q, k=3)])</code></pre><p>첫 두 질문은 A형 1.1판 1쪽을 찾아야 합니다. 마지막 질문은 기종이 빠져 두 기종 중 어느 동작을 안내할지 확정할 수 없습니다. 같은 후보 순위와 답의 확실성은 별도로 판단합니다.</p><p>집필 시 실행 결과</p><pre>2027년 5월 A형 녹음 표시등을 끌 수 있나요? [&#x27;recorder-a-1.1.pdf#p1&#x27;, &#x27;recorder-a-1.0.pdf#p1&#x27;, &#x27;recorder-b-1.0.pdf#p1&#x27;]
2027년 5월 A형 녹음 중 앞의 빨간불을 감출 수 있나요? [&#x27;recorder-a-1.1.pdf#p1&#x27;, &#x27;recorder-a-1.0.pdf#p1&#x27;, &#x27;recorder-a-1.1.pdf#p2&#x27;]
녹음 중에 앞쪽 불을 감출 수 있나요? [&#x27;recorder-b-1.0.pdf#p2&#x27;, &#x27;recorder-a-1.1.pdf#p2&#x27;, &#x27;recorder-b-1.0.pdf#p1&#x27;]</pre></details>

## 독립 실습 · 문의자가 제품명도 모를 때

“앞에 불빛이 들어오지 않도록 설정하고 싶어요”를 검색합니다. 답할 문장을 찾기 전에 검색된 A형과 B형 각각의 동작을 쓰세요. 이어 “2027년 5월 B형”을 추가해 검색합니다. **순위가 어떻게 바뀌었는지**와 **어떤 안내를 확정할 수 있게 되었는지**는 다른 칸에 씁니다. 자기 기종을 모르는 학생에게 보낼 확인 질문 한 문장을 작성하세요.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>A형은 녹음 중 표시등을 끌 수 없고 B형은 녹음 전에 표시등 숨김을 선택할 수 있습니다. 기종이 없으면 “가지고 계신 기기가 A형인가요, B형인가요?”라고 묻습니다. 상위에 B형이 가장 먼저 나왔다는 관찰은 질문자가 B형을 사용한다는 근거가 아닙니다.</p></details>

## 다음 수업에 가져갈 것

실제 질문·상위 세 ID·근거 문장을 남기세요. 다음 수업에서는 커널을 닫은 뒤에도 같은 색인을 열고, 다른 도구에서 **같은 벡터**를 비교합니다.